In [ ]:
from pathlib import Path
import json

import tifffile
import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF


# ------------------------------------------------------------
# Config
# ------------------------------------------------------------

ANNOTATION_PATH = Path("dataset/quadtree_index.json")
CNN_INPUT_SIZE = 512

In [ ]:
class QuadtreeWSIPairDataset(Dataset):
    def __init__(self, index_path, input_height=344, input_width = 512):
        self.index_path = Path(index_path)
        self.input_height = input_height
        self.input_width = input_width

        with open(self.index_path, "r", encoding="utf-8") as f:
            self.tile_jobs = json.load(f)

        # cache is per Dataset instance / per worker
        self._page_cache = {}

    def __len__(self):
        return len(self.tile_jobs)

    def _load_page_array(self, path, pyramid_page_idx):
        key = (str(path), int(pyramid_page_idx))

        if key not in self._page_cache:
            with tifffile.TiffFile(path) as slide:
                self._page_cache[key] = slide.pages[pyramid_page_idx].asarray()

        return self._page_cache[key]

    def _crop_tile(self, img, x_idx, y_idx, grid):
        H, W = img.shape[:2]

        tile_w = W // grid
        tile_h = H // grid

        x0 = x_idx * tile_w
        y0 = y_idx * tile_h

        x1 = W if x_idx == grid - 1 else x0 + tile_w
        y1 = H if y_idx == grid - 1 else y0 + tile_h

        return img[y0:y1, x0:x1]

    def _resize_to_tensor(self, tile):
        if tile.dtype != np.uint8:
            tile = tile.astype(np.uint8)

        image = Image.fromarray(tile)
        image = image.resize(
            (self.input_height, self.input_width),
            resample=Image.BILINEAR,
        )

        return TF.to_tensor(image)  # CHW float32 in [0, 1]

    def __getitem__(self, idx):
        job = self.tile_jobs[idx]

        fixed_page = self._load_page_array(
            path=job["fixed_path"],
            pyramid_page_idx=job["pyramid_page_idx"],
        )

        moving_page = self._load_page_array(
            path=job["moving_path"],
            pyramid_page_idx=job["pyramid_page_idx"],
        )

        fixed_tile = self._crop_tile(
            img=fixed_page,
            x_idx=job["x_idx"],
            y_idx=job["y_idx"],
            grid=job["grid"],
        )

        moving_tile = self._crop_tile(
            img=moving_page,
            x_idx=job["x_idx"],
            y_idx=job["y_idx"],
            grid=job["grid"],
        )

        fixed_tensor = self._resize_to_tensor(fixed_tile)
        moving_tensor = self._resize_to_tensor(moving_tile)

        transform = torch.tensor(
            job["transformation_matrix"],
            dtype=torch.float32,
        )

        registration_error = torch.tensor(
            job["registration_error"],
            dtype=torch.float32,
        )

        meta = {
            "pair_id": job["pair_id"],
            "source_image_id": job["source_image_id"],
            "target_image_id": job["target_image_id"],
            "crop_depth": job["crop_depth"],
            "grid": job["grid"],
            "x_idx": job["x_idx"],
            "y_idx": job["y_idx"],
            "pyramid_page_idx": job["pyramid_page_idx"],
            "tile_h": job["tile_h"],
            "tile_w": job["tile_w"],
        }

        return {
            "fixed": fixed_tensor,
            "moving": moving_tensor,
            "transform": transform,
            "registration_error": registration_error,
            "meta": meta,
        }

In [ ]:
dataset = QuadtreeWSIPairDataset(
    index_path=ANNOTATION_PATH,
    input_size=CNN_INPUT_SIZE,
)

loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=False,      # full deterministic traversal through the index
    num_workers=0,      # start with 0; later try 2
    pin_memory=True,
)

print("dataset length:", len(dataset))

dataset length: 7848


In [5]:
batch = next(iter(loader))

print(batch["fixed"].shape)
print(batch["moving"].shape)
print(batch["transform"].shape)
print(batch["registration_error"].shape)
print(batch["meta"])

torch.Size([4, 3, 512, 512])
torch.Size([4, 3, 512, 512])
torch.Size([4, 3, 3])
torch.Size([4])
{'pair_id': tensor([0, 0, 0, 0]), 'source_image_id': tensor([6036, 6036, 6036, 6036]), 'target_image_id': tensor([6045, 6045, 6045, 6045]), 'crop_depth': tensor([0, 1, 1, 1]), 'grid': tensor([1, 2, 2, 2]), 'x_idx': tensor([0, 0, 1, 0]), 'y_idx': tensor([0, 0, 0, 1]), 'pyramid_page_idx': tensor([4, 4, 4, 4]), 'tile_h': tensor([1388,  694,  694,  694]), 'tile_w': tensor([2054, 1027, 1027, 1027])}


/Users/alexanderhallmann/Desktop/medical-image-registration/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
